# Error Case 27 — 500/503: PostgreSQL Max Connections Exceeded

**Error:** `500 Internal Server Error` / `503 Service Unavailable`  
**Root cause:** All PostgreSQL connections consumed — new connection attempts fail  
**Fix:** Reduce pool size, add PgBouncer, or increase PostgreSQL max_connections

## Flow
```
Polaris pool: max-size=300 (your config)
PostgreSQL: max_connections=100 (default)
  → Polaris tries to open 300 connections
  → PostgreSQL: "sorry, too many clients already"
  → Agroal: cannot create connection → pool exhausted
  → Requests: 500 "Unable to acquire connection"
```

## Your current config risk
```
Polaris values.yaml:
  QUARKUS_DATASOURCE_JDBC_MAX_SIZE: 300

PostgreSQL default max_connections: 100
  → Polaris alone exceeds PostgreSQL limit
  → If other services share PostgreSQL → worse
  → Must match pool size to PostgreSQL capacity
```

In [1]:
# ── src path bootstrap (shared modules live in /src) ────────────────────────────
import sys, pathlib
_SRC = pathlib.Path.cwd().parent / "src"
if str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import sys
import importlib, polaris_test_utils
importlib.reload(polaris_test_utils)
from polaris_test_utils import *
require_not_prod("error-case test")  # PROD-safe: refuses to run against company PROD
import subprocess

cleanup()
print('✅ Setup complete')

🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🟢 LOCAL   POLARIS_ENV=local   (Polaris 1.3.0)
   Polaris: http://192.168.139.2:8181
   MinIO:   http://192.168.139.2:9000  bucket=data-catalog-bucket
   OpenSearch: localhost:9200  index=k8s-logs-*
   purge_deletes_files=False
🧹 Cleaning up all test resources...
✅ Cleanup done
✅ Setup complete


In [2]:
# Cell 1.5: Provision the DB Connection Pool OpenSearch monitor (idempotent)
# Same Monitor 7 from error-cases/opensearch-monitor-alert-guide.md as
# error-cases/15 — one monitor covers both root causes (pgpool down entirely,
# and pool exhaustion here). Registering it again is a no-op if 15 already
# ran (idempotent: looks up by name and updates in place).
import opensearch_alert_provisioner as alert_prov

try:
    db_pool_monitor_id = alert_prov.provision_db_connection_pool_monitor()
    print(f'✅ Monitor ready: {db_pool_monitor_id}')
except RuntimeError as e:
    db_pool_monitor_id = None
    print(f'⚠️  Skipping monitor provisioning: {e}')
    print('   (Create the "Polaris SMS Webhook" destination manually first — guide Step 1)')

⚠️  Skipping monitor provisioning: Notification destination "Polaris SMS Webhook" not found. Create it manually first (guide Step 1, Alerting -> Notifications -> Create channel) before provisioning the monitor — this module intentionally does not create destinations pointed at the internal Kafka webhook.
   (Create the "Polaris SMS Webhook" destination manually first — guide Step 1)


In [3]:
# Cell 2: Check current PostgreSQL max_connections and active connections
def pg_query(sql):
    """Run query via pgpool → primary."""
    # Find primary pod
    for i in range(3):
        pod = f'benchmarks-postgresql-postgresql-ha-postgresql-{i}'
        result = subprocess.run([
            'kubectl', 'exec', '-n', 'datahub-hynix', pod,
            '--', 'env', 'PGPASSWORD=polaris',
            'psql', '-U', 'polaris', '-d', 'polaris',
            '-c', 'SELECT pg_is_in_recovery();'
        ], capture_output=True, text=True)
        if 'f' in result.stdout:
            r = subprocess.run([
                'kubectl', 'exec', '-n', 'datahub-hynix', pod,
                '--', 'env', 'PGPASSWORD=polaris',
                'psql', '-U', 'polaris', '-d', 'polaris', '-c', sql
            ], capture_output=True, text=True)
            return r.stdout
    return 'Primary not found'

print('=== PostgreSQL connection stats ===')
print(pg_query('SHOW max_connections;'))
print(pg_query("""
SELECT count(*) as active_connections,
       max_conn.setting::int as max_connections,
       max_conn.setting::int - count(*) as available
FROM pg_stat_activity,
     pg_settings max_conn
WHERE max_conn.name = 'max_connections'
GROUP BY max_conn.setting;
"""))

print('=== Connections by application ===')
print(pg_query("""
SELECT application_name, count(*) as connections
FROM pg_stat_activity
WHERE state IS NOT NULL
GROUP BY application_name
ORDER BY connections DESC;
"""))

=== PostgreSQL connection stats ===
 max_connections 
-----------------
 100
(1 row)


 active_connections | max_connections | available 
--------------------+-----------------+-----------
                 23 |             100 |        77
(1 row)


=== Connections by application ===
    application_name    | connections 
------------------------+-------------
 psql                   |           6
 PostgreSQL JDBC Driver |           6
(2 rows)




In [4]:
# Cell 3: Simulate connection exhaustion
# Temporarily reduce max_connections to simulate exhaustion
import threading, time

print('Simulating connection pool pressure...')
print('Sending concurrent requests to exhaust pool...')

errors = []
success = []
lock = threading.Lock()

def hammer_request(req_id):
    try:
        r = requests.get(
            f'{BASE_MGMT}/catalogs',
            headers=h(),
            timeout=10
        )
        with lock:
            if r.status_code == 200:
                success.append(req_id)
            else:
                errors.append({'id': req_id, 'status': r.status_code,
                               'msg': r.json().get('error', {}).get('message', '')[:60]})
    except Exception as e:
        with lock:
            errors.append({'id': req_id, 'status': 'exception', 'msg': str(e)[:60]})

# Send 50 concurrent requests
N = 50
threads = [threading.Thread(target=hammer_request, args=(i,)) for i in range(N)]
for t in threads: t.start()
for t in threads: t.join()

print(f'\n{N} concurrent requests:')
print(f'  Success: {len(success)}')
print(f'  Errors:  {len(errors)}')
if errors:
    for e in errors[:3]:
        print(f'    {e["status"]}: {e["msg"]}')

Simulating connection pool pressure...
Sending concurrent requests to exhaust pool...

50 concurrent requests:
  Success: 49
  Errors:  1
    exception: Expecting value: line 1 column 1 (char 0)


In [5]:
# Cell 4: Check for connection exhaustion logs
wait_for_logs(10)
print('Searching for connection pool errors...')
hits = search_db_errors(minutes_ago=2)
print_logs(hits)

# Also check blocked thread (connection wait)
hits_blocked = search_blocked_thread_errors(minutes_ago=2)
if hits_blocked:
    print('\nBlocked thread warnings (connection wait):')
    print_logs(hits_blocked)

⏳ Waiting 10s for logs to reach OpenSearch...
Searching for connection pool errors...
Found 10 log entries:

⚠️  [2026-07-03T04:47:14.902Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': FATAL: Sorry, too many clients already

⚠️  [2026-07-03T04:47:14.902Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': FATAL: Sorry, too many clients already

⚠️  [2026-07-03T04:47:14.902Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': FATAL: Sorry, too many clients already

⚠️  [2026-07-03T04:47:14.902Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': FATAL: Sorry, too many clients already

⚠️  [2026-07-03T04:47:14.902Z] WARN
   logger    : io.agroal.pool
   requestId : 
   realmId   : 
   message   : Datasource '<default>': FATAL: Sorry, too many clients already

⚠️  [2

In [ ]:
# Cell 4.5: Assert the DB Connection Pool ALERT trigger actually fires (Case B — pool exhaustion)
# Force-runs the monitor now (dryrun=True — evaluates the trigger condition
# without sending a real SMS/Messenger notification on every test run) and
# checks the ALERT trigger by name (trigger_fired() scans trigger_results by
# name, since results are keyed by trigger ID). This is the second of the
# two DB-connection-pool scenarios the tightened (2026-07-03) monitor query
# has to catch: not the DB being down (error-cases/15), but PostgreSQL being
# reachable while max_connections itself is exhausted — the real captured
# signal is the raw PostgreSQL "too many clients already" message relayed
# through Agroal.
if db_pool_monitor_id:
    result = alert_prov.execute_monitor_now(db_pool_monitor_id, dryrun=True)
    fired = alert_prov.trigger_fired(result, alert_prov.ALERT_TRIGGER_NAME)
    print(f'Monitor execution result: {result}')
    print(f'\nALERT trigger fired: {fired}')
    if not fired:
        print('⚠️  ALERT did NOT fire — check whether the 50-request hammer above actually')
        print('   produced pool-exhaustion errors (some environments have enough headroom)')
    else:
        print('✅ ALERT trigger confirmed firing on pool exhaustion (dryrun — no notification sent)')
    print('   Note: this proves detection only. Actual SMS/Messenger delivery to the')
    print('   internal Kafka webhook cannot be confirmed from this sandbox — verify manually')
    print('   once the real destination URL is wired in (see guide Step 1).')
    print('   RESOLVED-trigger behavior is validated once in error-cases/15 (shared monitor).')
else:
    print('⚠️  Skipped — monitor was not provisioned (see Cell 1.5)')

In [7]:
# Cell 5: Check current pool config vs PostgreSQL capacity
print('=== Connection pool analysis ===')

# Polaris pool config
result = subprocess.run([
    'kubectl', 'exec', '-n', 'datahub-hynix',
    'deployment/benchmarks-polaris',
    '--', 'env'
], capture_output=True, text=True)
pool_vars = [l for l in result.stdout.split('\n') if 'JDBC' in l and 'SIZE' in l]
for v in pool_vars:
    print(f'  {v}')

# PostgreSQL max_connections
max_conn = pg_query('SHOW max_connections;')
print(f'\nPostgreSQL max_connections: {max_conn.strip()}')

print('\n=== Recommendation ===')
print('Rule: pool_size <= (max_connections - reserved) / num_polaris_pods')
print('Example:')
print('  PostgreSQL max_connections: 200')
print('  Reserved for admin/repmgr:  20')
print('  Available:                  180')
print('  Polaris pods (HPA max):     3')
print('  Pool size per pod:          180/3 = 60 (not 300!)')
print('')
print('Your current risk:')
print('  QUARKUS_DATASOURCE_JDBC_MAX_SIZE=300')
print('  But PostgreSQL likely has lower max_connections')
print('  Fix: reduce to 60-100 per pod')

=== Connection pool analysis ===
  QUARKUS_DATASOURCE_JDBC_MIN_SIZE=10
  QUARKUS_DATASOURCE_JDBC_MAX_SIZE=300

PostgreSQL max_connections: max_connections 
-----------------
 100
(1 row)

=== Recommendation ===
Rule: pool_size <= (max_connections - reserved) / num_polaris_pods
Example:
  PostgreSQL max_connections: 200
  Reserved for admin/repmgr:  20
  Available:                  180
  Polaris pods (HPA max):     3
  Pool size per pod:          180/3 = 60 (not 300!)

Your current risk:
  QUARKUS_DATASOURCE_JDBC_MAX_SIZE=300
  But PostgreSQL likely has lower max_connections
  Fix: reduce to 60-100 per pod


In [8]:
# Cell 6: Fix recommendation — values.yaml changes
print('Fix 1: Reduce pool size in values.yaml')
print()
print('extraEnv:')
print('  # Reduce max pool to safe level')
print('  - name: QUARKUS_DATASOURCE_JDBC_MAX_SIZE')
print('    value: "60"   # was 300')
print()
print('  # Keep min connections reasonable')
print('  - name: QUARKUS_DATASOURCE_JDBC_MIN_SIZE')
print('    value: "5"    # was 10')
print()
print('  # Fail fast when pool exhausted')
print('  - name: QUARKUS_DATASOURCE_JDBC_ACQUISITION_TIMEOUT')
print('    value: "PT5S"')
print()
print('Fix 2: Increase PostgreSQL max_connections')
print()
print('  kubectl exec -n datahub-hynix \\')
print('    benchmarks-postgresql-postgresql-ha-postgresql-0 --')
print('    psql -U postgres -c')
print('    "ALTER SYSTEM SET max_connections=500;"')
print()
print('Fix 3: Add PgBouncer (production recommended)')
print('  → Connection pooling at PostgreSQL level')
print('  → Polaris connects to PgBouncer, not PostgreSQL directly')
print('  → PgBouncer multiplexes: 300 Polaris → 50 PostgreSQL')

cleanup()
print('\n=== Summary ===')
print('Error:    500 Unable to acquire connection')
print('Cause:    pool_size > PostgreSQL max_connections')
print('Detect:   search_db_errors() + search_blocked_thread_errors()')
print('Fix:      Reduce pool max-size OR increase PostgreSQL max_connections')
print('Formula:  pool_size = (pg_max_conn - reserved) / num_pods')
print('Best:     Add PgBouncer for production workloads')

Fix 1: Reduce pool size in values.yaml

extraEnv:
  # Reduce max pool to safe level
  - name: QUARKUS_DATASOURCE_JDBC_MAX_SIZE
    value: "60"   # was 300

  # Keep min connections reasonable
  - name: QUARKUS_DATASOURCE_JDBC_MIN_SIZE
    value: "5"    # was 10

  # Fail fast when pool exhausted
  - name: QUARKUS_DATASOURCE_JDBC_ACQUISITION_TIMEOUT
    value: "PT5S"

Fix 2: Increase PostgreSQL max_connections

  kubectl exec -n datahub-hynix \
    benchmarks-postgresql-postgresql-ha-postgresql-0 --
    psql -U postgres -c
    "ALTER SYSTEM SET max_connections=500;"

Fix 3: Add PgBouncer (production recommended)
  → Connection pooling at PostgreSQL level
  → Polaris connects to PgBouncer, not PostgreSQL directly
  → PgBouncer multiplexes: 300 Polaris → 50 PostgreSQL
🧹 Cleaning up all test resources...
✅ Cleanup done

=== Summary ===
Error:    500 Unable to acquire connection
Cause:    pool_size > PostgreSQL max_connections
Detect:   search_db_errors() + search_blocked_thread_errors()
Fi